# BTS Digital Twin - Round1 Public One-Click Resume Test

Notebook này chỉ để trả lời 1 câu hỏi:

- train `30000` iteration rồi đo điểm
- resume lên `60000` iteration rồi đo lại điểm
- nếu điểm tăng thật trên `round1 public_set`, mới mang workflow đó sang `round2`

Mục tiêu thao tác: chỉ cần dán `GITHUB_TOKEN` nếu repo private, rồi `Run All`.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!pip install -q gdown plyfile tqdm lpips scikit-image


## Bước 1 - Clone gaussian-splatting


In [ ]:
%cd /kaggle/working
!rm -rf gaussian-splatting
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/simple-knn
import os
os.environ['GS_REPO'] = '/kaggle/working/gaussian-splatting'
print('GS_REPO =', os.environ['GS_REPO'])


## Bước 2 - Clone repo pipeline


In [ ]:
REPO_URL = 'https://github.com/ThongLuc2k3/BTS-Digital-Twin.git'
GIT_BRANCH = 'main'
GITHUB_TOKEN = ''

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
        print('Đã lấy GITHUB_TOKEN từ Kaggle Secrets')
except Exception:
    pass

clone_url = REPO_URL
if GITHUB_TOKEN and 'github.com' in REPO_URL:
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

!rm -rf /kaggle/working/project
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/project
%cd /kaggle/working/project


## Bước 3 - Chọn scene và tự dò dataset round1

Notebook sẽ tự tìm `VAI_NVS_DATA/phase1/public_set` trong:
- `/kaggle/working/project/Dataset`
- `/kaggle/input/...`

Nếu không tìm thấy, khi đó mới cần tự gắn dataset vào Kaggle hoặc sửa tay `DATASET_ROOT_OVERRIDE`.


In [ ]:
SCENE = 'hcm0031'  # hcm0031 | hcm0034 | HCM0181 | HCM0193 | HCM0204
DATASET_ROOT_OVERRIDE = ''

import os
from pathlib import Path

expected = {'hcm0031', 'hcm0034', 'HCM0181', 'HCM0193', 'HCM0204'}
candidates = []

if DATASET_ROOT_OVERRIDE:
    candidates.append(Path(DATASET_ROOT_OVERRIDE))

candidates.append(Path('/kaggle/working/project/Dataset/VAI_NVS_DATA/phase1/public_set'))

for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
    if base.exists():
        for p in base.rglob('public_set'):
            try:
                names = {x.name for x in p.iterdir() if x.is_dir()}
            except Exception:
                continue
            if expected <= names:
                candidates.append(p)

DATASET_ROOT = None
for p in candidates:
    if p.is_dir():
        names = {x.name for x in p.iterdir() if x.is_dir()}
        if expected <= names:
            DATASET_ROOT = str(p)
            break

assert DATASET_ROOT, 'Không tìm thấy Dataset/VAI_NVS_DATA/phase1/public_set. Gắn dataset vào Kaggle hoặc điền DATASET_ROOT_OVERRIDE.'
os.environ['DATASET_ROOT'] = DATASET_ROOT
print('DATASET_ROOT =', DATASET_ROOT)
print('SCENE =', SCENE)


## Bước 4 - Train 30000 và chấm điểm


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work' / SCENE
MODEL_DIR = WORK_DIR / 'gs_model'
RENDER_DIR_30K = WORK_DIR / 'round1_test_renders_30000'
WORK_DIR.mkdir(parents=True, exist_ok=True)

os.environ['ITERATIONS'] = '30000'
os.environ['ANTIALIASING'] = '1'
os.environ['EXPOSURE_COMP'] = '1'
os.environ['SAVE_FINAL_CHECKPOINT'] = '1'
os.environ.pop('START_CHECKPOINT', None)

subprocess.run(['bash', str(PROJECT / 'pipeline' / 'scripts' / '03_train_3dgs.sh'), SCENE], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'render_round1_test_poses.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--model_dir', str(MODEL_DIR),
    '--iteration', '30000',
    '--out_dir', str(RENDER_DIR_30K),
], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'eval_round1_metrics.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--renders_dir', str(RENDER_DIR_30K),
    '--out_csv', str(WORK_DIR / 'eval_round1_metrics_30000.csv'),
], check=True)


## Bước 5 - Resume 60000 và chấm lại điểm


In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work' / SCENE
MODEL_DIR = WORK_DIR / 'gs_model'
RENDER_DIR_60K = WORK_DIR / 'round1_test_renders_60000'
START_CKPT = MODEL_DIR / 'chkpnt30000.pth'
assert START_CKPT.exists(), f'Không thấy checkpoint resume: {START_CKPT}'

os.environ['ITERATIONS'] = '60000'
os.environ['ANTIALIASING'] = '1'
os.environ['EXPOSURE_COMP'] = '1'
os.environ['SAVE_FINAL_CHECKPOINT'] = '1'
os.environ['START_CHECKPOINT'] = str(START_CKPT)

subprocess.run(['bash', str(PROJECT / 'pipeline' / 'scripts' / '03_train_3dgs.sh'), SCENE], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'render_round1_test_poses.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--model_dir', str(MODEL_DIR),
    '--iteration', '60000',
    '--out_dir', str(RENDER_DIR_60K),
], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'eval_round1_metrics.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--renders_dir', str(RENDER_DIR_60K),
    '--out_csv', str(WORK_DIR / 'eval_round1_metrics_60000.csv'),
], check=True)


## Bước 6 - So sánh score 30000 và 60000


In [ ]:
import csv
from pathlib import Path

WORK_DIR = Path('/kaggle/working/project/pipeline/work') / SCENE
csv_30k = WORK_DIR / 'eval_round1_metrics_30000.csv'
csv_60k = WORK_DIR / 'eval_round1_metrics_60000.csv'

def mean_score(path):
    rows = list(csv.DictReader(open(path)))
    return sum(float(r['score']) for r in rows) / len(rows)

score_30k = mean_score(csv_30k)
score_60k = mean_score(csv_60k)
print(f'Score 30000: {score_30k:.6f}')
print(f'Score 60000: {score_60k:.6f}')
print(f'Chênh lệch : {score_60k - score_30k:+.6f}')
if score_60k > score_30k:
    print('Kết luận: resume có cải thiện trên round1 public_set.')
else:
    print('Kết luận: resume chưa cho cải thiện trên round1 public_set.')


## Bước 7 - Nếu muốn dùng lại checkpoint

Giữ nguyên thư mục này:

`/kaggle/working/project/pipeline/work/<SCENE>/gs_model`


In [ ]:
print(f'/kaggle/working/project/pipeline/work/{SCENE}/gs_model')
